# How to Run Ingest a PDF into the Weaviate database in the Cloud
This notebook goes over an example case of scraping a PDF, chunking it, uploading it to a Weaviate database, and querying the database. It is designed to be run on data-int.lsst.cloud, and you must have access to that platform in order to successfully run it. While this example case is for a PDF, this workflow is designed to be used for any data source, so long as said data source can be converted to a Langchain document object.

In [ ]:
%pip install weaviate-client
%pip install langchain==0.3.20
%pip install openai==1.65.4
%pip install langchain-openai==0.3.7
%pip install langchain-weaviate==0.0.4
%pip install langchain-community==0.3.19
%pip install pymupdf
%pip install python-dotenv

In [1]:
import os
import time
from dotenv import load_dotenv

import weaviate
from weaviate.classes.init import Auth
from weaviate.connect import ConnectionParams
from weaviate.classes.config import Configure, DataType, Property
from weaviate.classes.query import MetadataQuery

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_weaviate.vectorstores import WeaviateVectorStore
from langchain_core.documents.base import Document

## Connect to Weaviate
Test the connection to weaviate. Make sure to have a .env file in the same directory with OPENAI_API_KEY and WEAVIATE_API_KEY set to the correct values.

In [2]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
weaviate_api_key = os.getenv("WEAVIATE_API_KEY")
http_host = "weaviate-headless.rubin-rag.svc.cluster.local"
grpc_host = "weaviate-grpc.rubin-rag.svc.cluster.local"

if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable is not set")
if weaviate_api_key is None:
    raise ValueError("WEAVIATE_API_KEY environment variable is not set")
if http_host is None:
    raise ValueError("HTTP_HOST environment variable is not set")
if grpc_host is None:
    raise ValueError("GRPC_HOST environment variable is not set")

client = weaviate.connect_to_custom(
    http_host=http_host,
    http_port=8080,  # Default is 80, WCD uses 443
    http_secure=False,
    grpc_host=grpc_host,
    grpc_port=50051,  # Default is 50051, WCD uses 443
    grpc_secure=False,
    auth_credentials=Auth.api_key(
        weaviate_api_key
    ),  # The API key to use for authentication
    headers={"X-OpenAI-Api-Key": openai_api_key},
    skip_init_checks=True,
)

print("Client is live:", client.is_live())
print(client.collections.list_all().keys()) # List all collections in database
# print(client.collections.get("collection_name")) # View the configuration of a collection
# client.collections.delete("collection_name")  # THIS WILL DELETE THE SPECIFIED COLLECTION AND ITS OBJECTS
client.close()

Client is live: True
dict_keys(['Full_PDF_Ingestion', 'LangChain_9787ec4b92d3438a8de3ff04ead7ead6', 'PDF_Ingestion_Test_5', 'Parent_Test'])


## Scrape PDFs
To test locally, put PDFs in a directory named "pdfs" at the same level as this notebook. This part should output a list of Langchain Document objects, one for each PDF you put in the directory.  

In [3]:
def load_and_scrape(pdf_directory):
    """Load and scrape a PDF into a langchain document object."""
    
    pdf_files = [f for f in os.listdir(pdf_directory) if f.endswith(".pdf")]

    if not pdf_files:
        print("No PDFs found in the directory.")
        return
    
    documents = []
    
    for pdf_file in pdf_files:
        pdf_path = os.path.join(pdf_directory, pdf_file)
        print(f"Loading PDF: {pdf_file}")
        
        loader = PyMuPDFLoader(pdf_path)
        document = loader.load()
        documents += document
        
    return documents

In [4]:
# Test the scraper
# Note: this will print every document, so maybe test on a smaller doc first
pdf_directory = "./pdfs/"
documents = load_and_scrape(pdf_directory=pdf_directory)
for doc in documents:
    print(doc.page_content)

Loading PDF: LSST_Overview_Paper.pdf


<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


Loading PDF: LSST_SRD.pdf
Draft version August 13, 2019
Typeset using LATEX twocolumn style in AASTeX62
LSST: from Science Drivers to Reference Design and Anticipated Data Products
ˇZeljko Ivezi´c,1 Steven M. Kahn,2, 3 J. Anthony Tyson,4 Bob Abel,5 Emily Acosta,2 Robyn Allsman,2
David Alonso,6 Yusra AlSayyad,7 Scott F. Anderson,1 John Andrew,2 James Roger P. Angel,8
George Z. Angeli,9 Reza Ansari,10 Pierre Antilogus,11 Constanza Araujo,2 Robert Armstrong,7
Kirk T. Arndt,6 Pierre Astier,11 ´Eric Aubourg,12 Nicole Auza,2 Tim S. Axelrod,8 Deborah J. Bard,13
Jeff D. Barr,2 Aurelian Barrau,14 James G. Bartlett,12 Amanda E. Bauer,2 Brian J. Bauman,15
Sylvain Baumont,16, 11 Andrew C. Becker,1 Jacek Becla,13 Cristina Beldica,17 Steve Bellavia,18
Federica B. Bianco,19, 20 Rahul Biswas,21 Guillaume Blanc,10, 22 Jonathan Blazek,23, 24 Roger D. Blandford,3
Josh S. Bloom,25 Joanne Bogart,3 Tim W. Bond,13 Anders W. Borgland,13 Kirk Borne,26 James F. Bosch,7
Dominique Boutigny,27 Craig A. Brackett,13

## Chunk documents
This next step will chunk the documents. It will also output a list of langchain Documents, but this list will be longer.

In [5]:
from langchain_core.documents.base import Document
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

def chunk_docs(docs: list[Document], 
               chunk_size: int = 1000, 
               chunk_overlap: int = 50,
) -> list[Document]:
    """Chunk langchain documents.
    Parameters
    ----------
        docs : list 
            name of the list of langchain documents
        chunk_size : int
            size of chunks (in characters)
        chunk_overlap : int
            overlap of chunks (in characters)
    Returns
    -------
        docs : list
            list of langchain documents
        """

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = text_splitter.split_documents(docs)
    return chunks

In [6]:
# Test document chunker by printing out the first two chunks
chunked_docs = chunk_docs(docs=documents,
                          chunk_size=1000, 
                          chunk_overlap=50,
                         )
for n in range(2):
    print(chunked_docs[n])

page_content='Draft version August 13, 2019
Typeset using LATEX twocolumn style in AASTeX62
LSST: from Science Drivers to Reference Design and Anticipated Data Products
ˇZeljko Ivezi´c,1 Steven M. Kahn,2, 3 J. Anthony Tyson,4 Bob Abel,5 Emily Acosta,2 Robyn Allsman,2
David Alonso,6 Yusra AlSayyad,7 Scott F. Anderson,1 John Andrew,2 James Roger P. Angel,8
George Z. Angeli,9 Reza Ansari,10 Pierre Antilogus,11 Constanza Araujo,2 Robert Armstrong,7
Kirk T. Arndt,6 Pierre Astier,11 ´Eric Aubourg,12 Nicole Auza,2 Tim S. Axelrod,8 Deborah J. Bard,13
Jeff D. Barr,2 Aurelian Barrau,14 James G. Bartlett,12 Amanda E. Bauer,2 Brian J. Bauman,15
Sylvain Baumont,16, 11 Andrew C. Becker,1 Jacek Becla,13 Cristina Beldica,17 Steve Bellavia,18
Federica B. Bianco,19, 20 Rahul Biswas,21 Guillaume Blanc,10, 22 Jonathan Blazek,23, 24 Roger D. Blandford,3
Josh S. Bloom,25 Joanne Bogart,3 Tim W. Bond,13 Anders W. Borgland,13 Kirk Borne,26 James F. Bosch,7' metadata={'producer': 'pdfTeX-1.40.17', 'creator': 'L

## Upload to Weaviate
The last step of ingestion is to upload the documents to Weaviate. For this last step, you must reconnect to Weaviate. It is best to put that connection in a `try:` block, along with your other code. This way you can ensure the client will close even if an Error occurs.

In [7]:
def push_docs_to_weaviate(
    chunked_docs: list[Document], index_name: str
) -> None:

    embeddings = OpenAIEmbeddings(
        openai_api_key=openai_api_key,
        model="text-embedding-3-large" # Default model text-embedding-ada-002
    )
    WeaviateVectorStore.from_documents(documents=chunked_docs, 
                                       embedding=embeddings, 
                                       index_name=index_name,
                                       client=client,
                                       text_key="page_content",
                                       attributes=list(chunked_docs[0].metadata.keys()))

### Test by Creating a Sample Database

In [8]:
try:
    client = weaviate.connect_to_custom(
    http_host=http_host,
    http_port=8080,
    http_secure=False,
    grpc_host=grpc_host,
    grpc_port=50051,
    grpc_secure=False,
    auth_credentials=Auth.api_key(
        weaviate_api_key
    ),
    headers={"X-OpenAI-Api-Key": openai_api_key},
    skip_init_checks=True,
    )
    push_docs_to_weaviate(chunked_docs, "Full_PDF_Ingestion")
    print(client.collections.list_all().keys())
except Exception as e:
    print("Error:", e)
finally:
    client.close()

dict_keys(['Full_PDF_Ingestion', 'LangChain_9787ec4b92d3438a8de3ff04ead7ead6', 'PDF_Ingestion_Test_5', 'Parent_Test'])


### Test that Database was Populated
Warning: This will print out your entire database, so start with a small example

In [9]:
try:
    client = weaviate.connect_to_custom(
    http_host=http_host,
    http_port=8080,
    http_secure=False,
    grpc_host=grpc_host,
    grpc_port=50051,
    grpc_secure=False,
    auth_credentials=Auth.api_key(
        weaviate_api_key
    ),
    headers={"X-OpenAI-Api-Key": openai_api_key},
    skip_init_checks=True,
    )
    collection_name = "Full_PDF_Ingestion"
    if collection_name not in client.collections.list_all():
        print(f"Collection '{collection_name}' does not exist. Run ingestion first.")

    collection = client.collections.get(collection_name)
    for item in collection.iterator():
        print(item.uuid, item.properties)
except Exception as e:
    print("Error:", e)
finally:
    client.close()

005038bc-4652-4f3d-bc8a-936ed7e702de {'subject': '', 'creator': 'LaTeX with hyperref package', 'page_content': 'is measured using a calibration photodiode whose quan-\ntum eﬃciency is known to high accuracy. In addition,\nthe LSST system will explicitly measure the atmospheric\ntransmission spectrum associated with each image ac-\nquired.\nA dedicated 1.2-meter auxiliary calibration\ntelescope will obtain spectra of standard stars in LSST\nﬁelds, calibrating the atmospheric throughput as a func-\ntion of wavelength (Stubbs et al. 2007, see Figs. 5 and\n6). The LSST auxiliary telescope will take data at lower\nspectral resolution (R ∼150) but wider spectral cov-\nerage (340nm — 1.05µm) than shown in these ﬁgures,\nusing a slitless spectrograph and an LSST corner-raft\nCCD. Celestial spectrophotometric standard stars can\nbe used as a separate means of photometric calibration,\nalbeit only through the comparison of band-integrated\nﬂuxes with synthetic photometry calculations.\nA similar

## Query the Database
Now, we can query the Weaviate database to test whether the RAG is working. Use the collection name you specified above (for this example I use 'Full_PDF_Ingestion'), and make sure it is printed in the "Test by Creating a Sample Database was Created" section, so that you know it has been ingested.

### Keyword Search
This first test will just search for a given keyword in your database.

In [10]:
def bm25_search(keyword, collection_name):
    collection = client.collections.get(collection_name)
    response = collection.query.bm25(
        query=keyword,
        limit=4
    )
    
    for o in response.objects:
        print(o.properties)  # Object properties

In [11]:
try:
    client = weaviate.connect_to_custom(
    http_host=http_host,
    http_port=8080,
    http_secure=False,
    grpc_host=grpc_host,
    grpc_port=50051,
    grpc_secure=False,
    auth_credentials=Auth.api_key(
        weaviate_api_key
    ),
    headers={"X-OpenAI-Api-Key": openai_api_key},
    skip_init_checks=True,
    )
    bm25_search(keyword= "optimal", 
                collection_name= "Full_PDF_Ingestion")
except Exception as e:
    print("Error:", e)
finally:
    client.close()

{'subject': '', 'creator': 'LaTeX with hyperref package', 'page_content': '2. Image quality;\n3. Photometric accuracy;\n4. Astrometric accuracy;\n5. Optimal exposure time;\n6. The ﬁlter complement;\n7. The distribution of revisit times (i.e., the cadence\nof observations), including the survey lifetime;\n8. The total number of visits to a given area of sky;\n9. The coadded survey depth;\n10. The distribution of visits on the sky, and the total\nsky coverage;\n11. The distribution of visits per ﬁlter; and\n12. Parameters characterizing data processing and\ndata access (such as the maximum time allowed\nafter each exposure to report transient sources,\nand the maximum allowed software contribution\nto measurement errors).', 'total_pages': 57.0, 'keywords': '', 'modDate': 'D:20190813194917Z', 'trapped': '', 'text': None, 'format': 'PDF 1.5', 'creationdate': datetime.datetime(2019, 8, 13, 19, 49, 17, tzinfo=datetime.timezone.utc), 'file_path': './pdfs/LSST_Overview_Paper.pdf', 'moddate': d

### RAG Query
Now we can do the final test: a RAG based query using the generative AI query function provided by weaviate.

In [12]:
def query_rag(question, collection_name):
    """Query Weaviate using the near vector search"""
    if collection_name not in client.collections.list_all():
        print(f"Collection '{collection_name}' does not exist. Run ingestion first.")
        return

    embedding_model = OpenAIEmbeddings(openai_api_key=openai_api_key, model="text-embedding-3-large")
    query_vector = embedding_model.embed_query(question)

    collection = client.collections.get(collection_name)
    # print(collection.config.get())

    response = collection.generate.near_vector(
        near_vector=query_vector,
        limit=4,
        grouped_task=question,
        return_metadata=MetadataQuery(distance=True),
)
    # print("RAW SEARCH RESPONSE:", response) # Uncomment this to see what chunks are pulled
    print("RAG GENERATED RESPONSE:", response.generated)

In [13]:
try:
    client = weaviate.connect_to_custom(
    http_host=http_host,
    http_port=8080,
    http_secure=False,
    grpc_host=grpc_host,
    grpc_port=50051,
    grpc_secure=False,
    auth_credentials=Auth.api_key(
        weaviate_api_key
    ),
    headers={"X-OpenAI-Api-Key": openai_api_key},
    skip_init_checks=True,
    )
    
    query_rag(question= "How many strongly lensed Type Ia supernovae is LSST expected to discover?", 
              collection_name= "Full_PDF_Ingestion")
except Exception as e:
    print("Error:", e)
finally:
    client.close()

RAG GENERATED RESPONSE: LSST is expected to discover between 500 and 1000 strongly lensed Type Ia supernovae.
